# Task 29 Manual Cross Validation

This notebook contains direct, self-contained code for the task.


In [ ]:
from pathlib import Path
import sys
import numpy as np

active = Path.cwd().resolve()
tasks_root = active if (active / 'manual_ml.py').exists() else active.parent
if str(tasks_root) not in sys.path:
    sys.path.insert(0, str(tasks_root))

from manual_ml import LogisticRegressionScratch, load_data, standardize, standardize_fit

possible = [active / 'heart.csv', tasks_root / 'heart.csv', active.parent / 'heart.csv']
csv_path = next((p for p in possible if p.exists()), Path('heart.csv'))
X, y, features, df = load_data(csv_path)
folds = np.array_split(np.arange(len(y)), 5)
accuracies = []
for idx, test_idx in enumerate(folds):
    train_idx = np.concatenate([part for i, part in enumerate(folds) if i != idx])
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    means, scales = standardize_fit(X_train)
    X_train_s = standardize(X_train, means, scales)
    X_test_s = standardize(X_test, means, scales)
    model = LogisticRegressionScratch(learning_rate=0.05, iterations=1200)
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    acc = float((preds == y_test).mean())
    accuracies.append(acc)
    print(f'Fold {idx + 1}: accuracy={acc:.3f}')
print('Mean CV accuracy:', round(float(np.mean(accuracies)), 4))
